# notebook 20 — 計量等方は与件由来か格子由来か：循環の検証

nb19 は「連続極限の計量レベルで連続回転 SO(3) 対称が創発する」と結論した。だがその構成は
暗黙に**立方格子**（3軸を等係数・直交で組む）を仮定していた。ここに循環の疑いがある:
*等方性を出力するために、すでに等方的な（立方対称な）軸配置を入力していた*のではないか。

本 notebook はこの疑いを正面から詰める。問いは明確:**計量等方は与件 cos2θ の帰結か、それとも
私が選んだ立方格子の任意性の帰結か**。前者なら循環でない、後者なら nb19 の結論は弱まる。

**規律（D-10）**：自分の前回結論（nb19）を弱めうる方向の検証なので、結論を守る誘惑に逆らい、
循環があるなら正直に認める。検証は軸配置（係数・直交性）を*変えて*計量がどう壊れるかを見る。

## 20.0 セットアップ

In [1]:
import numpy as np
import sympy as sp
np.set_printoptions(precision=5, suppress=True, linewidth=120)
N = 3

def rot(axis, ang):
    c, s = np.cos(ang), np.sin(ang)
    if axis == 0: return np.array([[1,0,0],[0,c,-s],[0,s,c]])
    if axis == 1: return np.array([[c,0,s],[0,1,0],[-s,0,c]])
    return np.array([[c,-s,0],[s,c,0],[0,0,1]])

def kern_weighted(v, w):
    s = 0.0
    for c, x in enumerate(v):
        a = abs(np.cos(2*x))/(2*N)
        s += w[c]*(-np.log(a))**2
    return s

print("setup done. N =", N)

setup done. N = 3


## 20.1 予想の数値確認：非等価軸で計量は異方になるか

各軸の結合強度を `w_c` で変え、距離核を `Σ_c w_c·d(Δφ_c)²` とする。`w_c` が揃わなければ2次項
（計量）が球対称でなくなるはず。小角度で回転による計量の破れを測る。

In [2]:
rng = np.random.default_rng(2)
amp = 0.05
print("  軸係数 w        計量の破れ /amp²     判定")
for w in [[1,1,1],[1,1,2],[1,2,3]]:
    errs = []
    for _ in range(2000):
        v = rng.normal(size=3); v = v/np.linalg.norm(v)*amp
        R = rot(rng.integers(3), rng.uniform(0, np.pi))
        errs.append(abs(kern_weighted(R@v, w) - kern_weighted(v, w)))
    rel = np.mean(errs)/amp**2
    verdict = "等方（計量不変）" if rel < 0.05 else "異方（計量で破れ）"
    print(f"  {str(w):12s}   {rel:10.4f}        {verdict}")
print()
print("  → 等係数 (1,1,1) のみ計量等方（残差は小角度高次）。係数を崩すと2次で異方。")
print("    『計量が等方』は『3軸が等価な係数で組まれた』ことに依存する。")

  軸係数 w        計量の破れ /amp²     判定
  [1, 1, 1]          0.0024        等方（計量不変）
  [1, 1, 2]          1.2914        異方（計量で破れ）
  [1, 2, 3]          2.6256        異方（計量で破れ）

  → 等係数 (1,1,1) のみ計量等方（残差は小角度高次）。係数を崩すと2次で異方。
    『計量が等方』は『3軸が等価な係数で組まれた』ことに依存する。


## 20.2 解析：計量テンソルは軸配置で決まる

数値を解析で裏づける。単一軸核の2次係数は nb19 より `2x²`。重み付き核の計量テンソルを書き下す。

In [3]:
x1, x2, x3 = sp.symbols("x1 x2 x3", real=True)
w1, w2, w3 = sp.symbols("w1 w2 w3", positive=True)
g = w1*2*x1**2 + w2*2*x2**2 + w3*2*x3**2
print(f"  重み付き計量（2次項）= {g}")
print("  → 計量テンソル g = diag(2w1, 2w2, 2w3)。等方 ⟺ w1 = w2 = w3。")
print("    『計量が等方』は数学的に『3軸が等係数』と同値。")

  重み付き計量（2次項）= 2*w1*x1**2 + 2*w2*x2**2 + 2*w3*x3**2
  → 計量テンソル g = diag(2w1, 2w2, 2w3)。等方 ⟺ w1 = w2 = w3。
    『計量が等方』は数学的に『3軸が等係数』と同値。


## 20.3 核心：3軸が等価であるべき根拠は与件にあるか

ここが循環判定の本丸。`w_c` が全軸で等しい根拠が**与件 cos2θ にある**なら循環でない、
**私の任意選択**なら循環。与件の性質を吟味する。

In [4]:
print("与件 cos2θ は『1種類』の相関関数：どの測定方向でも同じ cos2(相対角)。")
print()
print("→ 各軸が同じ核 d(Δφ) を持つ（w_c が全軸で等しい）のは、")
print("  『すべての測定方向が同一の与件相関に従う』という与件の【等質性】の帰結。")
print("  これは立方格子を手で入れたのではなく、与件が方向に依らないことから来る。")
print()
print("【循環判定 その1】軸の等価性（w_c 一定）= 与件の等質性 → 循環でない。")

与件 cos2θ は『1種類』の相関関数：どの測定方向でも同じ cos2(相対角)。

→ 各軸が同じ核 d(Δφ) を持つ（w_c が全軸で等しい）のは、
  『すべての測定方向が同一の与件相関に従う』という与件の【等質性】の帰結。
  これは立方格子を手で入れたのではなく、与件が方向に依らないことから来る。

【循環判定 その1】軸の等価性（w_c 一定）= 与件の等質性 → 循環でない。


## 20.4 反証可能性チェック：軸が非等価になる物理状況はあるか

「与件由来」という主張が空虚でないことを、反証可能性で確認する。軸が非等価になりうる物理状況を
想定し、与件がそうなっていないことを示す。

In [5]:
print("反証可能な状況：測定方向ごとに相関が違えば（方向依存の cos2θ）w_c は割れ計量は異方。")
print("  例：cos2θ の振幅や周期が方向で変わる与件 → 計量異方。")
print()
print("だが実際の与件はそうでない：cos2θ は相対角のみの関数で、絶対的な方向に依らない。")
print("→ 計量等方は『与件が方向等質』という反証可能な性質の帰結。任意選択ではない。")
print("  （もし将来 方向依存の相関が与件になれば、計量は異方になる——予言的内容を持つ。）")

反証可能な状況：測定方向ごとに相関が違えば（方向依存の cos2θ）w_c は割れ計量は異方。
  例：cos2θ の振幅や周期が方向で変わる与件 → 計量異方。

だが実際の与件はそうでない：cos2θ は相対角のみの関数で、絶対的な方向に依らない。
→ 計量等方は『与件が方向等質』という反証可能な性質の帰結。任意選択ではない。
  （もし将来 方向依存の相関が与件になれば、計量は異方になる——予言的内容を持つ。）


## 20.5 残る循環の芽：直交性は与件由来か

等価性（係数）は晴れた。残るは**直交性**。3軸を斜交（非直交）に組んだ場合を検討する。

In [6]:
print("軸を斜交（非直交）に組むと、計量テンソルは基底の Gram 行列 G になる。")
print("  G ∝ 単位行列 ⟺ 直交かつ等長。斜交なら G は非対角成分を持つ。")
print()
print("だが計量テンソルは座標変換で対角化でき、適当な座標では常に等方形にできる。")
print("→ 物理的不変量（曲率・固有値）は基底の取り方に依らない。")
print("  直交性は『座標の選択』の問題で、物理には影響しない。")
print()
print("【循環判定 その2】直交性は座標選択（gauge）→ 物理的循環ではない。")
print("  物理的に意味があるのは『等質性』（20.3、与件由来）。")

軸を斜交（非直交）に組むと、計量テンソルは基底の Gram 行列 G になる。
  G ∝ 単位行列 ⟺ 直交かつ等長。斜交なら G は非対角成分を持つ。

だが計量テンソルは座標変換で対角化でき、適当な座標では常に等方形にできる。
→ 物理的不変量（曲率・固有値）は基底の取り方に依らない。
  直交性は『座標の選択』の問題で、物理には影響しない。

【循環判定 その2】直交性は座標選択（gauge）→ 物理的循環ではない。
  物理的に意味があるのは『等質性』（20.3、与件由来）。


## 20.6 結論の三層分離

循環の疑いを三つの要素に分解すると、与件由来・外部入力・座標選択にきれいに分かれる。

In [7]:
print("="*60)
print("計量等方を支える3要素の素性：")
print("="*60)
print(" 1. 軸の等価性（w_c 一定）  : 与件 cos2θ の等質性の帰結 → 与件由来（循環でない）")
print(" 2. 軸の本数（3本）         : nb17 の外部入力（次元3）→ 与件外（未解決の最深 open）")
print(" 3. 軸の直交性             : 座標選択（gauge）→ 物理的不変量に影響せず")
print("="*60)
print()
print("→ nb19 の『計量等方』のうち、等方性そのもの（軸が等価）は与件由来で循環でない。")
print("  循環の疑いは晴れた。ただし『3本』であること（次元3）は依然 nb17 の外部入力。")
print("  nb19 の結論は『次元3を認めれば、計量等方は与件由来』と精密化される。")

計量等方を支える3要素の素性：
 1. 軸の等価性（w_c 一定）  : 与件 cos2θ の等質性の帰結 → 与件由来（循環でない）
 2. 軸の本数（3本）         : nb17 の外部入力（次元3）→ 与件外（未解決の最深 open）
 3. 軸の直交性             : 座標選択（gauge）→ 物理的不変量に影響せず

→ nb19 の『計量等方』のうち、等方性そのもの（軸が等価）は与件由来で循環でない。
  循環の疑いは晴れた。ただし『3本』であること（次元3）は依然 nb17 の外部入力。
  nb19 の結論は『次元3を認めれば、計量等方は与件由来』と精密化される。


## 20.7 サマリー（established / open）

### established（この notebook で確定）

1. **計量等方は軸の等価性（w_c 一定）に依存する（20.1–20.2）。** 重み付き核 `Σ w_c d(Δφ_c)²` の
   計量テンソルは `diag(2w_c)`。等方 ⟺ 全軸等係数。係数を崩すと2次（計量）レベルで異方が出る。

2. **軸の等価性は与件 cos2θ の等質性の帰結（20.3–20.4）。** 与件は1種類の相関関数で、どの測定方向でも
   同じ cos2(相対角)。全軸が同じ核を持つ（w_c 一定）のは「全方向が同一の与件相関に従う」等質性から
   来る。立方格子の任意選択ではない。方向依存の相関なら計量は異方になる（反証可能＝予言的）。

3. **直交性は座標選択で物理に影響しない（20.5）。** 斜交基底でも計量は座標変換で対角化でき、
   物理的不変量は基底に依らない。直交性は gauge の問題。

4. **循環の三層分離（20.6、核心成果）。** nb19 の計量等方を支える3要素は素性が異なる:
   *等価性*（与件由来、循環でない）／*軸数3*（nb17 の外部入力、未解決）／*直交性*（座標選択）。
   **循環の疑いは晴れた**——等方性そのものは与件由来。ただし「3本」であることは依然外部入力。

### nb19 結論の精密化

| nb19 の主張 | nb20 後の精密化 |
|---|---|
| 連続極限の計量レベルで SO(3) 対称が創発 | **次元3を認めれば**、計量等方は与件 cos2θ の等質性の帰結（循環でない） |
| 高次に立方異方性が残る | 立方異方性は軸を立方配置で組んだ高次の帰結（座標選択を超えた格子の痕跡） |

### open（次へ）

1. **「実空間3次元」の上流原理（最深 open、継続）。** nb20 で循環は晴れたが、軸数3（次元3）は
   依然 nb17 の外部入力。計量の等方性は与件由来でも、3次元であること自体は未導出のまま。
   これが本研究第一段階の最後の空白で、README §5 第二段階に属する。

2. **高次立方異方性と格子間隔 ε（nb19 open 継続）。** 直交性は座標選択だが、立方異方性（4次）は
   座標で消せない格子の痕跡。ε→0 でどう抑制されるかの定量関係は未解明。

3. **本丸 no-go の解析的裏づけ（C-1、継続）。**

### 教訓（D への追記候補）

- **D-21（候補）：自分の前回結論を弱めうる検証を率先せよ。** nb19 の「計量等方創発」には立方格子の
  循環の疑いがあった。それを守るのでなく正面から検証した結果、循環は晴れ（等価性は与件由来）、
  かつ結論が「次元3を認めれば」という正しい条件つきに精密化された。反証を試みることが結論を強くする。

- **D-22（候補）：複合した疑いは要素に分解して素性を個別判定せよ。** 「計量等方は格子由来か」という
  疑いは、等価性・軸数・直交性の3要素に分解すると、それぞれ与件由来・外部入力・座標選択と素性が
  異なった。一括で「循環/非循環」を問うと誤る。要素ごとに与件の内側/外側/gauge を判定する（D-14 の精緻化）。